# 16.11 編譯有何代價與收益？

# 第 16 章：由量測帶出效率 BKM

前置：已有第4章能forward與生成的模型。像整理廚房，先找哪一步最慢再改工作流程。CPU小例只驗證數值與資料契約；GPU速度、峯值記憶體與編譯收益必須在讀者的硬體實測。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：舊筆記不用重寫：只加新字的K/V**

cache保存每層筆記；不是提前知道下一字答案。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/cache.svg")))

**先想一想：**只量第一次compiled調用，可以推斷長跑速度嗎？

編譯先花時間分析與生成優化路徑，之後纔可能更快。短任務可能不回本；動態shape或Python分派可能觸發重編譯。

**把直覺寫成數學：**總時間=編譯開銷+N×穩態時間；N很小時未必有收益。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
layer = nn.Linear(4, 3)
x = torch.randn(2, 4)
compiled = torch.compile(layer, backend="eager")
assert torch.allclose(compiled(x), layer(x))
print("圖捕獲接口通過；backend=eager不是速度優化")
print("真實實驗另用默認backend，並記錄首跑、warmup、steady-state")

**如何讀結果：**eager避免Notebook意外長編譯，只檢查接口。實際GPU編譯需單獨入口，不用這個結果宣傳加速。

**只改一件事：**只改變輸入shape，觀察真實backend是否重新編譯。
